# MCP + Llama3.3 Demo

This notebook connects to the local MCP server (`http://localhost:8000/sse`), 
discovers its tools, and hands them to **Llama3.3** running in Ollama.

> **Prerequisites**
> 1. MCP server running: `cd .. && .venv/bin/python server.py`
> 2. Ollama running with llama3.3: `ollama serve`

## 1. Config

In [7]:
MCP_SERVER_URL = "http://localhost:8000/sse"
MODEL          = "llama3.1"

## 2. Imports

In [8]:
import ollama
from mcp.client.sse import sse_client
from mcp import ClientSession

## 3. Inspect MCP tools

Connect to the server and print what tools are registered.

In [9]:
async def list_mcp_tools():
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return result.tools

tools = await list_mcp_tools()
for t in tools:
    print(f"  tool: {t.name}")
    print(f"  desc: {t.description}")
    print(f"  schema: {t.inputSchema}")
    print()

  tool: hello_world
  desc: Returns a simple Hello World greeting.
  schema: {'properties': {}, 'title': 'hello_worldArguments', 'type': 'object'}



## 4. Chat with Llama3.3 + MCP tools

Full agentic loop:
1. Send user message + available tools to Llama3.3
2. If Llama requests a tool call, execute it via MCP
3. Feed the result back and get the final response

In [10]:
def mcp_tools_to_ollama(mcp_tools):
    """Convert MCP tool descriptors to Ollama's tool format."""
    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description,
                "parameters": t.inputSchema,
            },
        }
        for t in mcp_tools
    ]


async def chat(user_message: str) -> str:
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            mcp_tools  = (await session.list_tools()).tools
            ollama_tools = mcp_tools_to_ollama(mcp_tools)

            messages = [{"role": "user", "content": user_message}]
            print(f"User: {user_message}\n")

            # Agentic loop — keep going until no more tool calls
            while True:
                response = ollama.chat(
                    model=MODEL,
                    messages=messages,
                    tools=ollama_tools,
                )
                msg = response.message
                messages.append(msg)

                if not msg.tool_calls:
                    break  # Llama is done, return final text

                # Execute each requested tool via MCP
                for tc in msg.tool_calls:
                    name = tc.function.name
                    args = dict(tc.function.arguments or {})
                    print(f"  [tool call]  {name}({args})")

                    mcp_result = await session.call_tool(name, args)
                    text = mcp_result.content[0].text if mcp_result.content else ""
                    print(f"  [tool result] {text}\n")

                    messages.append({"role": "tool", "content": text})

            final = msg.content
            print(f"Llama3.1: {final}")
            return final

## 5. Run it

In [11]:
await chat("Please use the hello_world tool and tell me what it returned.")

User: Please use the hello_world tool and tell me what it returned.

  [tool call]  hello_world({})
  [tool result] Hello World!

Llama3.1: The `hello_world` tool called successfully and returned the expected output: 'Hello, World!'


"The `hello_world` tool called successfully and returned the expected output: 'Hello, World!'"